# 10. Time Series Analytics

**Machine Learning Fundamentals and Predictive Analytics — Notebook 10 of 11**

Every model so far assumed the rows were **independent** — shuffle them and nothing changes.
Time series data breaks that assumption on purpose: **order is the signal**. Yesterday's sales
predict today's; today's predicts tomorrow's.

This changes the rules for splitting, validating and modelling data (a theme first raised in
statistics Notebook 10). This notebook builds the toolkit from decomposition through to a
forecast with an honest, time-respecting evaluation.

### What you will learn

1. **Trend, seasonality, and residuals** — decomposing a series
2. **Stationarity**, and why most models need it
3. **Autocorrelation** (ACF/PACF) and what they reveal
4. **Moving averages** and **exponential smoothing**
5. **ARIMA**: AR, I, MA and how they combine
6. Modern **regression-based forecasting**: lag features + gradient boosting
7. **Time-respecting cross-validation** (no leakage from the future)
8. Forecast **evaluation**: RMSE, MAE, MAPE, and a naive baseline
9. **Anomaly detection** in time series

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller, acf, pacf
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.holtwinters import ExponentialSmoothing, SimpleExpSmoothing
from sklearn.model_selection import TimeSeriesSplit
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error

rng = np.random.default_rng(seed=10)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 4)

---
## 10.1 The anatomy of a time series

A time series is usually thought of as a sum (or product) of components:

$$y_t = \text{Trend}_t + \text{Seasonal}_t + \text{Residual}_t \qquad \text{(additive)}$$

$$y_t = \text{Trend}_t \times \text{Seasonal}_t \times \text{Residual}_t \qquad \text{(multiplicative)}$$

- **Trend** — the long-run direction (growing, shrinking, flat)
- **Seasonality** — a pattern that repeats at a **fixed, known** period (daily, weekly, yearly)
- **Residual** — what is left over: noise, and anything the model does not explain

**Additive vs multiplicative:** use additive when the seasonal swing stays roughly constant in
absolute size; multiplicative when it grows with the level (a retailer's December spike is
bigger in absolute terms as the business grows, but roughly the same *percentage*).

**Cyclic vs seasonal:** seasonality has a **fixed, known period** (always 7 days, always 12
months). A **cycle** (business cycles, epidemics) has no fixed length and is much harder to
model — decomposition tools do not handle it.

In [ ]:
# A realistic series: daily sales with trend, weekly seasonality, and noise
n_days = 3 * 365
t = np.arange(n_days)
trend = 100 + 0.06 * t + 15 * np.sin(2 * np.pi * t / 365)          # slow growth + a yearly wave
weekly = 20 * np.sin(2 * np.pi * t / 7 - 1.2) + 8 * (t % 7 >= 5)    # weekday/weekend pattern
noise = rng.normal(0, 6, n_days)
sales = np.maximum(trend + weekly + noise, 5)

dates = pd.date_range("2022-01-01", periods=n_days, freq="D")
ts = pd.Series(sales, index=dates, name="sales")

fig, ax = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
ax[0].plot(ts, lw=0.8, color="steelblue")
ax[0].set_title("Three years of daily sales")
ax[1].plot(ts["2023-01-01":"2023-02-28"], lw=1.4, color="steelblue")
ax[1].set_title("Zoomed in: the weekly pattern is visible")
plt.tight_layout(); plt.show()

In [ ]:
decomp = seasonal_decompose(ts, model="additive", period=7)

fig, axes = plt.subplots(4, 1, figsize=(11, 9), sharex=True)
for ax, comp, name in zip(axes, [ts, decomp.trend, decomp.seasonal, decomp.resid],
                          ["observed", "trend", "seasonal (period=7)", "residual"]):
    ax.plot(comp, lw=0.8, color="steelblue")
    ax.set_title(name, fontsize=10)
plt.tight_layout(); plt.show()

print(f"Residual std: {decomp.resid.std():.2f}  (compare to the injected noise sd = 6.0)")
print(f"Seasonal component range: [{decomp.seasonal.min():.2f}, {decomp.seasonal.max():.2f}]")
print("\nThe trend line recovers the growth AND the slow yearly wave (a period-7 decomposition")
print("cannot separate a 365-day cycle from 'trend' -- it has no way to know it exists).")
print("This is a real limitation: classical decomposition needs the periods you tell it about.")

---
## 10.2 Stationarity

Most classical time series models (ARIMA among them) assume the series is **stationary**: its
statistical properties do not change over time.

$$E[y_t] = \mu \ \ (\text{constant}), \qquad \operatorname{Var}(y_t) = \sigma^2\ \ (\text{constant}),
\qquad \operatorname{Cov}(y_t, y_{t+k}) \text{ depends only on } k$$

A series with a trend is not stationary (the mean drifts). A series with growing seasonal
swings is not stationary (the variance grows). Real data is almost never stationary as
collected — you make it stationary.

**Testing stationarity:** the **Augmented Dickey-Fuller (ADF) test**.

$$H_0:\ \text{the series has a unit root (non-stationary)} \qquad H_1:\ \text{stationary}$$

A small p-value (reject $H_0$) is evidence *for* stationarity — the opposite convention from
most tests you have used, so it trips people up.

**Making a series stationary: differencing.** $y_t' = y_t - y_{t-1}$ removes a linear trend.
Apply it again ($y_t'' = y_t' - y'_{t-1}$) for a quadratic trend; **seasonal differencing**
($y_t - y_{t-s}$) removes a repeating pattern of period $s$.

In [ ]:
def adf_report(series, name):
    result = adfuller(series.dropna())
    print(f"{name}")
    print(f"  ADF statistic = {result[0]:.4f}   p-value = {result[1]:.4f}")
    print(f"  {'STATIONARY (reject H0)' if result[1] < 0.05 else 'NON-STATIONARY (fail to reject H0)'}")
    return result[1]

adf_report(ts, "Raw series")
diff1 = ts.diff().dropna()
adf_report(diff1, "\nAfter first differencing")
diff_seasonal = ts.diff(7).dropna()
adf_report(diff_seasonal, "\nAfter seasonal differencing (lag 7)")
diff_both = ts.diff().diff(7).dropna()
adf_report(diff_both, "\nAfter BOTH first and seasonal differencing")

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(11, 9), sharex=False)
for ax, series, title in zip(axes, [ts, diff1, diff_seasonal, diff_both],
                             ["raw (non-stationary: trending mean)",
                              "1st difference (mean stabilised, weekly pattern remains)",
                              "seasonal difference lag 7 (weekly pattern removed, trend remains)",
                              "both differences (looks stationary)"]):
    ax.plot(series, lw=0.7, color="steelblue")
    ax.axhline(series.mean(), color="crimson", lw=1, ls="--")
    ax.set_title(title, fontsize=9)
plt.tight_layout(); plt.show()
print("Differencing is the 'I' (integrated) in ARIMA -- you difference until the ADF test")
print("says stationary, model THAT series, then integrate (cumulative sum) the forecast back.")

---
## 10.3 Autocorrelation: ACF and PACF

**Autocorrelation** measures how correlated a series is with a lagged copy of itself.

**ACF (autocorrelation function)** at lag $k$: $\operatorname{Corr}(y_t, y_{t-k})$ — the *raw*
correlation, including indirect effects through intermediate lags.

**PACF (partial autocorrelation function)** at lag $k$: the correlation between $y_t$ and
$y_{t-k}$ **after removing** the effect of lags $1, \dots, k-1$. It isolates the *direct*
relationship.

These two plots are the classical diagnostic for choosing ARIMA's order:

| Pattern | Suggests |
|---|---|
| ACF cuts off sharply after lag $q$, PACF tails off | **MA(q)** |
| PACF cuts off sharply after lag $p$, ACF tails off | **AR(p)** |
| Both tail off gradually | **ARMA(p, q)** — mixed |
| A spike at lag 7 (or 12, or 24...) | Seasonality with that period |

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 7))
plot_acf(ts, lags=40, ax=axes[0, 0], title="ACF: raw series (dominated by trend)")
plot_pacf(ts, lags=40, ax=axes[0, 1], title="PACF: raw series")
plot_acf(diff_both, lags=40, ax=axes[1, 0], title="ACF: after differencing")
plot_pacf(diff_both, lags=40, ax=axes[1, 1], title="PACF: after differencing")
plt.tight_layout(); plt.show()

print("Raw series: ACF decays very slowly -- the classic fingerprint of a trend.")
print("After differencing: look for a spike at lag 7 -- what remains of the weekly rhythm")
print("after removing the trend. The shaded band is the 95% 'not significantly different")
print("from zero' region; bars poking outside it are the lags worth including in a model.")

---
## 10.4 Smoothing methods

### Moving average

$$\hat{y}_t = \frac{1}{k}\sum_{i=0}^{k-1} y_{t-i}$$

Simple, but every past point counts equally and old data can be very old. A larger window
smooths more but reacts more slowly to real changes.

### Exponential smoothing

Weight recent observations more, with weights decaying exponentially into the past:

$$\hat{y}_{t+1} = \alpha y_t + (1-\alpha)\hat{y}_t, \qquad \alpha \in (0, 1)$$

**Holt-Winters** extends this with a trend term and a seasonal term, giving three smoothing
parameters ($\alpha, \beta, \gamma$) fit to minimise forecast error.

In [ ]:
ma7 = ts.rolling(7).mean()
ma30 = ts.rolling(30).mean()
ses = SimpleExpSmoothing(ts, initialization_method="estimated").fit(smoothing_level=0.2)

fig, ax = plt.subplots(figsize=(11, 4.5))
window = ts["2023-06-01":"2023-09-30"]
ax.plot(window, lw=0.8, color="grey", alpha=0.6, label="raw")
ax.plot(ma7["2023-06-01":"2023-09-30"], lw=2, color="steelblue", label="7-day moving average")
ax.plot(ma30["2023-06-01":"2023-09-30"], lw=2, color="crimson", label="30-day moving average")
ax.legend(fontsize=8); ax.set_title("Larger windows smooth more, and lag more")
plt.tight_layout(); plt.show()

print(f"7-day MA sd of residual : {(ts - ma7).std():.3f}")
print(f"30-day MA sd of residual: {(ts - ma30).std():.3f}")
print("The 30-day average is smoother but systematically lags a rising trend -- it is")
print("still reporting last month's level while the series has already moved on.")

In [ ]:
hw = ExponentialSmoothing(ts, trend="add", seasonal="add", seasonal_periods=7,
                          initialization_method="estimated").fit()
print(f"Fitted Holt-Winters parameters:")
print(f"  alpha (level)    = {hw.params['smoothing_level']:.4f}")
print(f"  beta  (trend)    = {hw.params['smoothing_trend']:.4f}")
print(f"  gamma (seasonal) = {hw.params['smoothing_seasonal']:.4f}")

forecast_hw = hw.forecast(30)
fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(ts["2024-08-01":], lw=1, color="steelblue", label="observed")
ax.plot(hw.fittedvalues["2024-08-01":], lw=1, color="seagreen", ls="--", label="fitted")
ax.plot(forecast_hw, lw=2, color="crimson", label="30-day forecast")
ax.axvline(ts.index[-1], color="black", lw=1)
ax.legend(fontsize=8); ax.set_title("Holt-Winters: level + trend + weekly seasonality")
plt.tight_layout(); plt.show()

---
## 10.5 ARIMA

**ARIMA(p, d, q)** combines three ideas:

- **AR(p)** — AutoRegressive: today depends linearly on the last $p$ values
  $$y_t = c + \phi_1 y_{t-1} + \dots + \phi_p y_{t-p} + \varepsilon_t$$
- **I(d)** — Integrated: difference the series $d$ times to make it stationary first
- **MA(q)** — Moving Average: today depends on the last $q$ **forecast errors**
  $$y_t = c + \varepsilon_t + \theta_1\varepsilon_{t-1} + \dots + \theta_q\varepsilon_{t-q}$$

**SARIMA(p,d,q)(P,D,Q)$_s$** adds a seasonal AR/I/MA structure at period $s$ — what you need
for the weekly pattern here.

Choosing orders: use ACF/PACF as a starting guess, then let **AIC** (Akaike Information
Criterion, statistics Notebook 3) pick between candidates — lower is better, and it penalises
extra parameters.

In [ ]:
train_ts = ts[:-30]
test_ts = ts[-30:]

rows = []
for order in [(1, 1, 1), (2, 1, 1), (1, 1, 2), (2, 1, 2), (3, 1, 1)]:
    try:
        m = ARIMA(train_ts, order=order,
                  seasonal_order=(1, 1, 1, 7)).fit()
        rows.append({"order": order, "AIC": m.aic, "BIC": m.bic})
    except Exception as e:
        rows.append({"order": order, "AIC": np.nan, "BIC": np.nan})
aic_table = pd.DataFrame(rows).sort_values("AIC")
print(aic_table.round(2).to_string(index=False))
print(f"\nBest order by AIC: {aic_table.iloc[0]['order']}")

In [ ]:
best_order = aic_table.iloc[0]["order"]
sarima = ARIMA(train_ts, order=best_order, seasonal_order=(1, 1, 1, 7)).fit()
print(sarima.summary().tables[1])

fc = sarima.get_forecast(steps=30)
mean_fc = fc.predicted_mean
ci = fc.conf_int(alpha=0.05)

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(train_ts[-60:], lw=1, color="steelblue", label="training data")
ax.plot(test_ts, lw=1.5, color="black", label="actual (held out)")
ax.plot(mean_fc, lw=2, color="crimson", label="SARIMA forecast")
ax.fill_between(mean_fc.index, ci.iloc[:, 0], ci.iloc[:, 1], color="crimson", alpha=0.2,
                label="95% CI")
ax.legend(fontsize=8); ax.set_title(f"SARIMA{best_order}x(1,1,1,7): 30-day forecast")
plt.tight_layout(); plt.show()

print(f"\nRMSE  : {np.sqrt(mean_squared_error(test_ts, mean_fc)):.3f}")
print(f"MAE   : {mean_absolute_error(test_ts, mean_fc):.3f}")
print(f"MAPE  : {mean_absolute_percentage_error(test_ts, mean_fc)*100:.2f}%")

In [ ]:
# Residual diagnostics: a well-specified model leaves white noise behind
resid = sarima.resid[7:]
fig, axes = plt.subplots(1, 3, figsize=(15, 3.6))
axes[0].plot(resid, lw=0.7, color="steelblue")
axes[0].set_title("Residuals over time")
plot_acf(resid, lags=30, ax=axes[1], title="ACF of residuals")
axes[2].hist(resid, bins=30, color="steelblue", edgecolor="white")
axes[2].set_title("Residual distribution")
plt.tight_layout(); plt.show()

from statsmodels.stats.diagnostic import acorr_ljungbox
lb = acorr_ljungbox(resid, lags=[10], return_df=True)
print(lb.round(4))
print(f"\nLjung-Box p-value: {lb['lb_pvalue'].iloc[0]:.4f}")
print("H0: residuals are white noise (no leftover autocorrelation).")
print(f"{'Fail to reject -> model captured the structure well' if lb['lb_pvalue'].iloc[0] > 0.05 else 'Reject -> structure remains; reconsider the order'}")

---
## 10.6 Regression-based forecasting

Classical models like ARIMA are elegant but limited to one series and its own past. A more
flexible modern approach: turn forecasting into a **supervised regression problem** by building
**lag features**, then apply anything from the rest of this module — a gradient-boosted tree,
a random forest, or plain linear regression.

$$y_t = f(y_{t-1}, y_{t-7}, y_{t-14}, \text{day-of-week}, \text{month}, \dots) + \varepsilon_t$$

This makes it trivial to add **exogenous features** (a holiday flag, weather, promotions) that
ARIMA cannot easily use, and it opens the door to every non-linear model in this course.

In [ ]:
def make_features(series, lags=(1, 2, 3, 7, 14, 21), roll_windows=(7, 14)):
    df = pd.DataFrame({"y": series})
    for lag in lags:
        df[f"lag_{lag}"] = series.shift(lag)
    for w in roll_windows:
        df[f"rollmean_{w}"] = series.shift(1).rolling(w).mean()
        df[f"rollstd_{w}"] = series.shift(1).rolling(w).std()
    df["dow"] = series.index.dayofweek
    df["month"] = series.index.month
    df["is_weekend"] = (series.index.dayofweek >= 5).astype(int)
    return df.dropna()

feat = make_features(ts)
print(f"Feature table: {feat.shape[0]} rows, {feat.shape[1]-1} features")
print(feat.head(3).round(2).to_string())

In [ ]:
feat_train, feat_test = feat[:-30], feat[-30:]
X_cols = [c for c in feat.columns if c != "y"]

models_reg = {
    "linear regression": LinearRegression(),
    "random forest": RandomForestRegressor(n_estimators=300, min_samples_leaf=3,
                                           random_state=0),
    "gradient boosting": GradientBoostingRegressor(n_estimators=300, max_depth=3,
                                                   learning_rate=0.05, random_state=0),
}
preds_reg = {}
print(f"{'model':<20}{'RMSE':>9}{'MAE':>8}{'MAPE %':>9}")
for name, m in models_reg.items():
    m.fit(feat_train[X_cols], feat_train["y"])
    pred = m.predict(feat_test[X_cols])
    preds_reg[name] = pred
    print(f"{name:<20}{np.sqrt(mean_squared_error(feat_test.y, pred)):>9.3f}"
          f"{mean_absolute_error(feat_test.y, pred):>8.3f}"
          f"{mean_absolute_percentage_error(feat_test.y, pred)*100:>9.2f}")

naive_pred = feat_test["lag_7"]                     # 'same day last week'
print(f"{'naive (lag 7)':<20}{np.sqrt(mean_squared_error(feat_test.y, naive_pred)):>9.3f}"
      f"{mean_absolute_error(feat_test.y, naive_pred):>8.3f}"
      f"{mean_absolute_percentage_error(feat_test.y, naive_pred)*100:>9.2f}")
print(f"{'SARIMA (from 10.5)':<20}{np.sqrt(mean_squared_error(test_ts, mean_fc)):>9.3f}"
      f"{mean_absolute_error(test_ts, mean_fc):>8.3f}"
      f"{mean_absolute_percentage_error(test_ts, mean_fc)*100:>9.2f}")

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(feat_test.y, lw=1.6, color="black", label="actual")
for name, pred in preds_reg.items():
    ax.plot(feat_test.index, pred, lw=1.4, label=name)
ax.plot(feat_test.index, naive_pred, lw=1.2, ls="--", color="grey", label="naive (lag 7)")
ax.legend(fontsize=8); ax.set_title("Regression-based forecasts vs the naive baseline")
plt.tight_layout(); plt.show()

gb = models_reg["gradient boosting"]
imp = pd.Series(gb.feature_importances_, index=X_cols).sort_values(ascending=False)
print("Feature importances (gradient boosting):")
print(imp.round(4).to_string())
print("\nlag_7 dominates -- exactly what we would expect from a series with strong weekly")
print("seasonality. This is the regression model rediscovering, on its own, the same")
print("period-7 structure the ACF plot showed us directly.")

---
## 10.7 Time-respecting cross-validation

**Never shuffle a time series when splitting.** Training on future data to predict the past is
leakage, exactly as flagged in statistics Notebook 10 — here it is easy to commit by accident
because a plain `train_test_split(shuffle=True)` will do it silently.

`TimeSeriesSplit` performs an **expanding window**: each fold trains on everything up to a
point and validates on the period right after it, so training data always precedes validation
data.

In [ ]:
tscv = TimeSeriesSplit(n_splits=5, test_size=30)

fig, ax = plt.subplots(figsize=(11, 3.5))
for i, (tr, va) in enumerate(tscv.split(feat)):
    ax.scatter(feat.index[tr], [i]*len(tr), c="steelblue", marker="_", s=8)
    ax.scatter(feat.index[va], [i]*len(va), c="crimson", marker="_", s=8)
ax.set_yticks(range(5)); ax.set_yticklabels([f"fold {i+1}" for i in range(5)])
ax.set_title("TimeSeriesSplit: training window (blue) always precedes validation (red)")
plt.tight_layout(); plt.show()

print(f"{'fold':>6}{'train rows':>12}{'val rows':>10}{'val start':>14}")
for i, (tr, va) in enumerate(tscv.split(feat), 1):
    print(f"{i:>6}{len(tr):>12}{len(va):>10}{str(feat.index[va[0]].date()):>14}")

In [ ]:
# Honest CV for the gradient boosting model, vs the misleading shuffled version
from sklearn.model_selection import KFold, cross_val_score

gb_model = GradientBoostingRegressor(n_estimators=300, max_depth=3, learning_rate=0.05,
                                     random_state=0)

honest = -cross_val_score(gb_model, feat[X_cols], feat["y"], cv=tscv,
                          scoring="neg_root_mean_squared_error")
shuffled = -cross_val_score(gb_model, feat[X_cols], feat["y"],
                            cv=KFold(5, shuffle=True, random_state=0),
                            scoring="neg_root_mean_squared_error")

print(f"TimeSeriesSplit CV RMSE (honest)  : {honest.mean():.3f} +/- {honest.std():.3f}")
print(f"Shuffled KFold CV RMSE (leaky)    : {shuffled.mean():.3f} +/- {shuffled.std():.3f}")
print("\nThe shuffled version looks better because the model can 'interpolate' using")
print("neighbouring days that are, in a real deployment, still in the future. This is the")
print("time series version of every leakage warning from earlier in the course.")

---
## 10.8 Evaluating forecasts properly

| Metric | Formula | Notes |
|---|---|---|
| **RMSE** | $\sqrt{\text{mean}((y-\hat y)^2)}$ | Same units as $y$; penalises large misses |
| **MAE** | $\text{mean}(\|y-\hat y\|)$ | Robust, easy to explain |
| **MAPE** | $\text{mean}(\|\,(y-\hat y)/y\,\|)\times 100$ | Percent; **explodes near $y=0$** |
| **sMAPE** | symmetric MAPE | Bounded, less sensitive to small $y$ |

**Always report a naive baseline alongside your model:**

- **Naive** — forecast = last observed value
- **Seasonal naive** — forecast = the value from one season ago (here, 7 days ago)

A forecast that cannot beat "copy last week" has no business being deployed, however
sophisticated it looks. This mirrors the `DummyRegressor` discipline from the earlier
regression notebook.

In [ ]:
def evaluate_forecast(actual, pred, name):
    return {
        "model": name,
        "RMSE": np.sqrt(mean_squared_error(actual, pred)),
        "MAE": mean_absolute_error(actual, pred),
        "MAPE_%": mean_absolute_percentage_error(actual, pred) * 100,
    }

results = [
    evaluate_forecast(feat_test.y, feat_test["lag_1"], "naive (yesterday)"),
    evaluate_forecast(feat_test.y, feat_test["lag_7"], "seasonal naive (last week)"),
    evaluate_forecast(test_ts, mean_fc, "SARIMA"),
]
for name, pred in preds_reg.items():
    results.append(evaluate_forecast(feat_test.y, pred, name))
summary = pd.DataFrame(results).sort_values("RMSE")
print(summary.round(3).to_string(index=False))

best_rmse = summary.iloc[0].RMSE
snaive_rmse = summary[summary.model == "seasonal naive (last week)"].RMSE.item()
print(f"\nBest model beats seasonal naive by "
      f"{(1 - best_rmse/snaive_rmse)*100:.1f}% RMSE.")
print("Report THIS number to a stakeholder, not the raw RMSE -- it answers the question")
print("they actually asked: 'is your model worth building, compared to doing nothing clever?'")

---
## 10.9 Anomaly detection in time series

A common downstream use: flag points that deviate sharply from the expected pattern (fraud, a
sensor fault, a server outage). Three practical approaches, from simplest to most robust:

1. **Rolling z-score** — flag points many standard deviations from a rolling mean
2. **Seasonal decomposition residuals** — decompose, then flag large residuals (this correctly
   ignores expected seasonal swings, unlike a raw threshold)
3. **Forecast interval** — flag points outside a model's prediction interval (from SARIMA or a
   quantile regression)

In [ ]:
# Inject some anomalies into a fresh series
anomaly_ts = ts.copy()
anomaly_idx = rng.choice(len(anomaly_ts), 8, replace=False)
anomaly_ts.iloc[anomaly_idx] += rng.choice([-1, 1], 8) * rng.uniform(40, 70, 8)

# Method 1: rolling z-score
roll_mean = anomaly_ts.rolling(30, center=True).mean()
roll_std = anomaly_ts.rolling(30, center=True).std()
z = (anomaly_ts - roll_mean) / roll_std
flagged_z = np.abs(z) > 3

# Method 2: decomposition residuals
decomp2 = seasonal_decompose(anomaly_ts, model="additive", period=7)
resid2 = decomp2.resid
resid_z = (resid2 - resid2.mean()) / resid2.std()
flagged_resid = np.abs(resid_z) > 3

print(f"True anomalies injected : {len(anomaly_idx)}")
print(f"Flagged by rolling z-score        : {flagged_z.sum()}")
print(f"Flagged by decomposition residual  : {flagged_resid.sum()}")

true_mask = np.zeros(len(anomaly_ts), dtype=bool)
true_mask[anomaly_idx] = True
print(f"\nTrue positives (rolling z-score)  : {(flagged_z.fillna(False).values & true_mask).sum()}")
print(f"False positives (rolling z-score) : {(flagged_z.fillna(False).values & ~true_mask).sum()}")
print(f"True positives (decomposition)    : {(flagged_resid.fillna(False).values & true_mask).sum()}")
print(f"False positives (decomposition)   : {(flagged_resid.fillna(False).values & ~true_mask).sum()}")

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
ax[0].plot(anomaly_ts, lw=0.7, color="steelblue")
ax[0].scatter(anomaly_ts.index[flagged_z.fillna(False)],
              anomaly_ts[flagged_z.fillna(False)], color="crimson", s=50, zorder=5,
              label="flagged (rolling z-score)")
ax[0].scatter(anomaly_ts.index[anomaly_idx], anomaly_ts.iloc[anomaly_idx],
              facecolors="none", edgecolors="black", s=120, zorder=4, label="true anomaly")
ax[0].legend(fontsize=8); ax[0].set_title("Rolling z-score method")

ax[1].plot(anomaly_ts, lw=0.7, color="steelblue")
ax[1].scatter(anomaly_ts.index[flagged_resid.fillna(False)],
              anomaly_ts[flagged_resid.fillna(False)], color="crimson", s=50, zorder=5,
              label="flagged (decomposition residual)")
ax[1].scatter(anomaly_ts.index[anomaly_idx], anomaly_ts.iloc[anomaly_idx],
              facecolors="none", edgecolors="black", s=120, zorder=4, label="true anomaly")
ax[1].legend(fontsize=8); ax[1].set_title("Seasonal decomposition residual method")
plt.tight_layout(); plt.show()

print("A plain rolling z-score can be fooled by legitimate seasonal peaks (a busy Saturday")
print("is not an anomaly). Decomposing first and checking the RESIDUAL avoids that, because")
print("expected weekly swings are already subtracted out before the threshold is applied.")

---
## Exercises

**Exercise 1.** Simulate a monthly series with a multiplicative seasonal pattern (December
sales triple relative to a baseline that itself grows over time). Decompose it with both
`model="additive"` and `model="multiplicative"`, and show which one leaves cleaner residuals.

In [ ]:
# --- Solution 1 -------------------------------------------------------------
months = pd.date_range("2018-01-01", periods=84, freq="MS")
base = 100 + 1.5 * np.arange(84)                                  # growing baseline
season_mult = 1 + 0.6 * np.sin(2*np.pi*(np.arange(84) % 12)/12 - 1.4)
season_mult[np.arange(84) % 12 == 11] *= 1.8                      # December spike
sales_m = base * season_mult * (1 + rng.normal(0, 0.03, 84))
ts_m = pd.Series(sales_m, index=months)

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(ts_m, color="steelblue"); ax.set_title("Monthly sales: growing baseline, multiplicative spike")
plt.show()

d_add = seasonal_decompose(ts_m, model="additive", period=12)
d_mul = seasonal_decompose(ts_m, model="multiplicative", period=12)

print(f"Additive residual std       : {d_add.resid.std():.4f}")
print(f"Multiplicative residual std : {d_mul.resid.std():.4f}  (residuals are RATIOS near 1.0)")

fig, ax = plt.subplots(1, 2, figsize=(13, 3.8))
ax[0].plot(d_add.resid, color="crimson", lw=0.8); ax[0].axhline(0, color="black", lw=1)
ax[0].set_title("Additive residuals: still show a pattern")
ax[1].plot(d_mul.resid, color="seagreen", lw=0.8); ax[1].axhline(1, color="black", lw=1)
ax[1].set_title("Multiplicative residuals: cleaner, centred at 1")
plt.tight_layout(); plt.show()
print("\nThe additive residuals still swell around December, because a FIXED absolute term")
print("cannot represent a spike that scales with the baseline. The multiplicative model")
print("captures it correctly, since the seasonal factor multiplies rather than adds.")

**Exercise 2.** Fit a SARIMA model to the sales series using the last 60 days as a test set.
Try three different `(p,d,q)` combinations, compare by AIC and by out-of-sample RMSE, and check
whether they agree.

In [ ]:
# --- Solution 2 -------------------------------------------------------------
train2, test2 = ts[:-60], ts[-60:]
candidates = [(1, 1, 1), (2, 1, 0), (0, 1, 2), (2, 1, 2)]

rows2 = []
for order in candidates:
    m = ARIMA(train2, order=order, seasonal_order=(1, 1, 1, 7)).fit()
    fc2 = m.get_forecast(60).predicted_mean
    rows2.append({"order": order, "AIC": m.aic,
                  "test_RMSE": np.sqrt(mean_squared_error(test2, fc2))})
t2 = pd.DataFrame(rows2)
print(t2.round(2).to_string(index=False))
print(f"\nBest by AIC        : {t2.loc[t2.AIC.idxmin(), 'order']}")
print(f"Best by test RMSE   : {t2.loc[t2.test_RMSE.idxmin(), 'order']}")
print("\nThey usually agree closely, because AIC approximates out-of-sample error -- but")
print("they need not agree exactly, since AIC is an IN-SAMPLE, asymptotic approximation and")
print("the test RMSE is measured on one particular 60-day holdout. When they disagree,")
print("trust the held-out evaluation; it is the one that mirrors deployment.")

**Exercise 3.** Build lag features for the sales series and compare a plain linear regression,
random forest, and gradient boosting model using `TimeSeriesSplit`. Then add a holiday
indicator feature (mark 10 random dates as holidays with a demand spike) and show the
improvement.

In [ ]:
# --- Solution 3 -------------------------------------------------------------
holiday_dates = pd.DatetimeIndex(rng.choice(ts.index[30:], 10, replace=False))
ts_h = ts.copy()
ts_h.loc[holiday_dates] *= 1.6                                   # a demand spike on holidays

feat_h = make_features(ts_h)
feat_h["is_holiday"] = feat_h.index.isin(holiday_dates).astype(int)
feat_h["days_to_next_holiday"] = [
    min([abs((h - d).days) for h in holiday_dates] or [999]) for d in feat_h.index
]

tscv3 = TimeSeriesSplit(n_splits=5, test_size=40)
cols_no_holiday = [c for c in feat_h.columns if c not in ("y", "is_holiday",
                                                          "days_to_next_holiday")]
cols_with_holiday = cols_no_holiday + ["is_holiday", "days_to_next_holiday"]

print(f"{'model':<22}{'features':>10}{'CV RMSE':>10}")
for name, est in [("linear regression", LinearRegression()),
                  ("random forest", RandomForestRegressor(n_estimators=300, random_state=0)),
                  ("gradient boosting", GradientBoostingRegressor(n_estimators=300,
                                                                  max_depth=3,
                                                                  random_state=0))]:
    for cols, label in [(cols_no_holiday, "no holiday"), (cols_with_holiday, "with holiday")]:
        rmses = -cross_val_score(est, feat_h[cols], feat_h["y"], cv=tscv3,
                                 scoring="neg_root_mean_squared_error")
        print(f"{name:<22}{label:>10}{rmses.mean():>10.3f}")
print("\nAdding the holiday feature helps most for the tree-based models, which can use it")
print("directly as a split; linear regression benefits less because holidays interact")
print("multiplicatively with the base level, which a single additive term cannot capture")
print("(the same additive/multiplicative issue from Exercise 1).")

**Exercise 4 (challenge).** You are asked to forecast next month's revenue and to say how
confident the forecast is. Build a full pipeline: decompose, check stationarity, fit SARIMA
and a gradient-boosting lag model, evaluate both with `TimeSeriesSplit` against a seasonal
naive baseline, and produce a forecast with an uncertainty band you can defend.

In [ ]:
# --- Solution 4 -------------------------------------------------------------
print("STEP 1: decompose and check stationarity\n")
d_final = seasonal_decompose(ts, model="additive", period=7)
p_val = adf_report(ts, "Raw series")
p_val_d = adf_report(ts.diff().diff(7).dropna(), "\nDifferenced series")

In [ ]:
print("\nSTEP 2: fit both model families, evaluated with TimeSeriesSplit\n")
tscv4 = TimeSeriesSplit(n_splits=6, test_size=30)
feat4 = make_features(ts)

def eval_sarima_cv(series, order, seasonal_order, n_splits=6, horizon=30):
    errs = []
    for i in range(n_splits, 0, -1):
        cut = len(series) - i*horizon
        if cut < 100:
            continue
        tr, te = series[:cut], series[cut:cut+horizon]
        m = ARIMA(tr, order=order, seasonal_order=seasonal_order).fit()
        pred = m.get_forecast(len(te)).predicted_mean
        errs.append(np.sqrt(mean_squared_error(te, pred)))
    return np.array(errs)

sarima_cv = eval_sarima_cv(ts, (2, 1, 1), (1, 1, 1, 7))
gb_cv = -cross_val_score(GradientBoostingRegressor(n_estimators=300, max_depth=3,
                                                   learning_rate=0.05, random_state=0),
                         feat4.drop(columns="y"), feat4.y, cv=tscv4,
                         scoring="neg_root_mean_squared_error")
naive_cv = []
for tr_idx, va_idx in tscv4.split(feat4):
    naive_cv.append(np.sqrt(mean_squared_error(feat4.y.iloc[va_idx],
                                               feat4.lag_7.iloc[va_idx])))
naive_cv = np.array(naive_cv)

print(f"{'model':<24}{'mean RMSE':>11}{'sd':>8}")
print(f"{'seasonal naive':<24}{naive_cv.mean():>11.3f}{naive_cv.std():>8.3f}")
print(f"{'SARIMA(2,1,1)x(1,1,1,7)':<24}{sarima_cv.mean():>11.3f}{sarima_cv.std():>8.3f}")
print(f"{'gradient boosting':<24}{gb_cv.mean():>11.3f}{gb_cv.std():>8.3f}")

In [ ]:
print("\nSTEP 3: pick the winner and produce the final forecast with an uncertainty band\n")
winner = "SARIMA" if sarima_cv.mean() < gb_cv.mean() else "gradient boosting"
print(f"Winner by cross-validated RMSE: {winner}\n")

final_model = ARIMA(ts, order=(2, 1, 1), seasonal_order=(1, 1, 1, 7)).fit()
fc_final = final_model.get_forecast(30)
mean_final = fc_final.predicted_mean
ci_final = fc_final.conf_int(alpha=0.10)                        # 90% interval

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(ts[-90:], lw=1, color="steelblue", label="history")
ax.plot(mean_final, lw=2, color="crimson", label="30-day forecast")
ax.fill_between(mean_final.index, ci_final.iloc[:, 0], ci_final.iloc[:, 1],
                color="crimson", alpha=0.2, label="90% interval")
ax.legend(fontsize=8); ax.set_title("Final forecast with an honest uncertainty band")
plt.tight_layout(); plt.show()

monthly_total = mean_final[:30].sum()
lo_total = ci_final.iloc[:30, 0].sum()
hi_total = ci_final.iloc[:30, 1].sum()
print(f"Point forecast for next 30 days total : {monthly_total:,.0f}")
print(f"90% interval                          : [{lo_total:,.0f}, {hi_total:,.0f}]")
print(f"\nCross-validated RMSE for this model : {sarima_cv.mean():.2f}")
print(f"That beats the seasonal-naive baseline by "
      f"{(1 - sarima_cv.mean()/naive_cv.mean())*100:.1f}%.")
print()
print("What I would tell the stakeholder:")
print("  * point estimate, with an explicit interval -- never a bare number")
print("  * the model beats 'copy last week' by a stated, cross-validated margin")
print("  * the interval widens with the forecast horizon (visible in the fan shape); a")
print("    forecast for day 30 is less certain than for day 1, and the chart shows that")
print("  * the model assumes the recent pattern continues -- flag any known future change")
print("    (a promotion, a new store opening) that the model cannot see in historical data")

---
## Summary

| Concept | Key point |
|---|---|
| Decomposition | $y_t = \text{trend} + \text{seasonal} + \text{residual}$ (or product) |
| Stationarity | Constant mean/variance/autocovariance; test with ADF |
| Differencing | Removes trend (1st diff) or seasonality (lag-$s$ diff) |
| ACF / PACF | Diagnose AR vs MA order; a spike at lag $s$ signals seasonality |
| Moving average | Simple, but lags; window size trades smoothness for responsiveness |
| Exponential smoothing | Recent observations weighted more; Holt-Winters adds trend + seasonal |
| ARIMA(p,d,q) | AR + Integrated + MA; choose orders via ACF/PACF and AIC |
| SARIMA | Adds seasonal (P,D,Q)$_s$ terms |
| Lag-feature regression | Turns forecasting into ordinary supervised learning; adds exogenous features easily |
| `TimeSeriesSplit` | Never train on the future; expanding window |
| Naive / seasonal naive baseline | Always report improvement over "copy last period" |
| Metrics | RMSE, MAE, MAPE (careful near $y=0$) |
| Anomaly detection | Decompose first, then threshold the **residual**, not the raw series |

**Next up:** [Notebook 11 — Recommender Systems](11.%20Recommender%20Systems.ipynb), the final
notebook, where we predict not a number over time but *what a person will like*.